In [1]:
import polars as pl
import os

from pathlib import Path

In [9]:
DONOR = 'donor_4'

In [10]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/16_Pairwise Ranking Healthy Liver/')
DATASET_PATH = WORKING_PATH / 'dataset'/ DONOR
OUTPUT_PATH  = WORKING_PATH / 'output' / 'antisymmetric' / DONOR

In [11]:
### Merge the results into single CSV file

# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*.csv")

In [12]:
pl_df

seed,histone_marker,epochs_trained,train_loss_avg,train_accuracy_avg,train_auc_avg,val_loss_avg,val_accuracy_avg,val_auc_avg,test_accuracy,test_auc,antisymmetry
i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1011,"""H3K4me3""",39,0.591,72.2292,0.7233,0.5981,70.9949,0.7103,71.5,0.7148,0.888
123,"""H3K4me3""",60,0.5891,71.96,0.7192,0.5481,73.8833,0.739,71.6,0.7169,0.892
42,"""H3K4me3""",44,0.5919,71.7311,0.7166,0.5659,72.9864,0.733,71.8,0.7205,0.883
456,"""H3K4me3""",70,0.5784,72.945,0.7291,0.5546,73.5443,0.7397,70.0,0.7016,0.876
789,"""H3K4me3""",31,0.5945,72.0197,0.721,0.5858,70.5065,0.7087,72.2,0.7216,0.865
…,…,…,…,…,…,…,…,…,…,…,…
1011,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",53,0.5634,74.0962,0.7408,0.584,71.6925,0.7172,74.1,0.7409,0.935
123,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",61,0.5615,74.2223,0.7433,0.5262,74.5984,0.7469,74.3,0.7436,0.931
42,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",78,0.5582,74.1281,0.741,0.5289,75.3756,0.7541,74.7,0.7486,0.941


In [13]:
summary_df = (
    pl_df
    .group_by("histone_marker")
    .agg([
        pl.col("val_accuracy_avg").mean().alias("val_accuracy_mean"),
        pl.col("val_accuracy_avg").std().alias("val_accuracy_std"),
        pl.col("test_accuracy").mean().alias("test_accuracy_mean"),
        pl.col("test_accuracy").std().alias("test_accuracy_std"),
    ])
    .sort("test_accuracy_mean", descending=True)
)

In [14]:
summary_df

histone_marker,val_accuracy_mean,val_accuracy_std,test_accuracy_mean,test_accuracy_std
str,f64,f64,f64,f64
"""H3K9ac-H3K9me3-H3K27ac-H3K27me…",73.49722,1.586118,74.42,1.308434
"""H3K4me3-H3K9ac-H3K27ac-H3K27me…",73.85736,2.024634,74.3,0.685565
"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",73.8716,1.803229,74.18,0.782304
"""H3K9ac-H3K27ac-H3K27me3""",73.6301,1.410558,74.06,1.46731
"""H3K4me3-H3K9me3-H3K27ac""",73.60184,1.655615,73.98,0.729383
…,…,…,…,…
"""H3K4me3""",72.38308,1.533923,71.42,0.837854
"""H3K9ac-H3K9me3""",71.88428,1.754233,71.38,1.84716
"""H3K9me3-H3K27me3""",50.8642,1.26342,52.6,0.894427


In [15]:
summary_df.write_csv(OUTPUT_PATH/ f"{DONOR}.csv", include_header=True)